# ZiguratIP on Google Colab

Build and run **ZiguratIP** — a single-process database (*Zigurat*), language (*Parsi*),
and web server (*Zeytun*) written in dependency-free C++11 — inside a Colab VM, and reach
its HTTP front end from your browser.

- **Zigurat** — MVCC storage engine, binary protocol on port **2160**
- **Parsi** — SQL-like language compiled to a `.so` the server `dlopen`s
- **Zeytun** — HTTP server for static files and `.zt` pages on port **2190**

> ⚠️ **Still a throwaway sandbox — but not for the reasons this notice used to give.**
> The crypto anything reachable over the network uses is OpenSSL's now: SHA and HMAC, the
> reading of a peer's certificate, and the whole TLS engine. Keys seeded from `time(0)`,
> the network-reachable compiler, the out-of-bounds reads in the HTTP parser, the
> stack-smashing buffer on large files and the unbounded allocations are all fixed.
>
> What is left is enough to keep it off anything that matters: **cross-site scripting by
> construction** (nothing in the `ECHO` path escapes what it is given, and the demo pages
> write database columns straight into HTML), **no write-ahead log** (a hard kill mid-commit
> corrupts the store, with no recovery path), and **zlib 1.2.11** with two CVEs against it.
> `doc/outstanding.md` in the repository is the full list, each item with where it lives.
>
> Any key material generated *before* this branch is compromised and must be reissued —
> two `ca keygen` runs in the same second used to produce byte-identical private keys.
> The `ca` tool still issues certificates with the in-tree RSA; the randomness feeding it
> is fixed, but nothing the network reaches goes near that code.

**How it maps to Colab:** Colab is the *compute*. Google Drive is only *storage* — nothing
runs "on" Drive. We therefore **build and run on the VM's local disk** and use Drive purely
for an optional copy-in / copy-out snapshot of the data directory. Drive's FUSE mount is
hostile to the storage engine's random in-place writes and to `dlopen`, so the live database
must never sit on the mount.

## 1 · Build

> **Use the branch below, not `master`.** ZiguratIP did not build on Linux at all: fixed-width
> integer types missing `<cstdint>`, `htonl` and friends coming back as glibc macros, and two
> libraries that never declared what they link against. macOS hid all three. Worse, every recipe
> in the top-level `Makefile` is prefixed with `@-`, so failures were stepped over and the run
> still ended with `******* all done *******` — a clean checkout produced 2 of 14 libraries and
> no executables while reporting success. The branch fixes that; `master` will still appear to
> build and then have nothing to run.

Colab already ships `g++` and `make`; the `apt-get` line is just insurance.

In [ ]:
import os, subprocess

REPO   = 'https://github.com/saman-pasha/ZiguratIP.git'
BRANCH = 'colab'   # all fixes live here; master does not build on Linux
SRC    = '/content/ZiguratIP'

!apt-get -qq install -y build-essential >/dev/null 2>&1 || true

# Check the branch exists before cloning. A clone of a branch that is not
# there fails quietly enough that every later cell blames something else.
ls = subprocess.run(['git','ls-remote','--heads',REPO,BRANCH],
                    capture_output=True, text=True)
assert BRANCH in ls.stdout, f'branch {BRANCH!r} not found on the remote:\n{ls.stdout}{ls.stderr}'
print(f'branch {BRANCH} found')

# Clone, or bring an existing clone up to date. Re-running this notebook
# in a session that already cloned used to skip straight past here and
# rebuild whatever was fetched the first time, so a fix pushed since then
# never arrived and the symptom it fixed was still there.
if not os.path.isdir(SRC):
    !git clone --depth 1 -b $BRANCH $REPO $SRC
else:
    !git -C $SRC fetch --depth 1 origin $BRANCH
    !git -C $SRC checkout -B $BRANCH FETCH_HEAD
assert os.path.isfile(f'{SRC}/Makefile'), 'clone produced no tree'

head = subprocess.run(['git','-C',SRC,'log','-1','--format=%h %s'],
                      capture_output=True, text=True).stdout.strip()
print('building:', head)

# A rebuild has to see the new sources, and the object files from the
# previous run are older than nothing the Makefile knows to check.
!make -C $SRC clean >/dev/null 2>&1 || true

%cd /content/ZiguratIP
!make MODE=Release 2>&1 | tail -n 5

# make exits 0 even when projects fail -- every recipe in the top-level
# Makefile is prefixed with @-, so failures are stepped over and it still
# prints 'all done'. Check what was actually produced instead.
libs = sorted(f for f in os.listdir('home/lib') if f.endswith('.so'))
bins = sorted(os.listdir('home/bin'))
print(f'\nlibraries: {len(libs)} (expect 14)')
print(f'executables: {bins}')
missing = {'Test','ca','parsi','ziguratip'} - set(bins)
assert not missing and len(libs) >= 14, f'BUILD INCOMPLETE -- missing {missing or "libraries"}'
print('build OK')

## 2 · Runtime environment

`ZIGURATIP_HOME` is both the install prefix and the runtime home. On **Linux** the shared
libraries are found via `LD_LIBRARY_PATH` (not macOS's `DYLD_LIBRARY_PATH`). A C++ compiler
must also stay on `PATH` — Parsi pages are compiled to a `.so` *at request time*, not only
at build time.

In [ ]:
os.environ['ZIGURATIP_HOME']  = '/content/ZiguratIP/home'
os.environ['LD_LIBRARY_PATH'] = os.environ['ZIGURATIP_HOME'] + '/lib'
assert subprocess.call(['which', 'c++']) == 0, 'a C++ compiler must be on PATH for runtime Parsi compilation'
print('ZIGURATIP_HOME =', os.environ['ZIGURATIP_HOME'])
print('binaries:')
!ls -1 $ZIGURATIP_HOME/bin

## 3 · (Optional) restore data from Google Drive

Colab VMs are ephemeral — `home/data` is lost when the runtime recycles. To keep a database
between sessions, set `PERSIST = True`. We copy a saved snapshot from Drive **onto local disk
before starting** the server; we never run the engine directly on the Drive mount.

Leave `PERSIST = False` for a clean, throwaway run (the store is created on first use).

In [ ]:
PERSIST = False  # set True to keep the database across sessions via Drive
DRIVE_BACKUP = '/content/drive/MyDrive/ziguratip-backup/data'

if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.isdir(DRIVE_BACKUP):
        print('restoring data snapshot from Drive ...')
        !rm -rf $ZIGURATIP_HOME/data
        !cp -a "$DRIVE_BACKUP" $ZIGURATIP_HOME/data
        print('restored.')
    else:
        print('no Drive snapshot yet — a fresh store will be created on first use.')
else:
    print('PERSIST is off — using a throwaway store on local disk.')

## 4 · Build the demo objects

`demo/build.sh` compiles the demo tables and `.zt` pages with the **offline** `parsi` compiler.
It starts nothing — it just produces the shared objects the server will load.

Note this is the supported way to compile. Compiling *over the network* is refused by default
(`COMPILER/REMOTE_MODE`), because it runs a C++ compiler and linker on whatever a client sends,
which is arbitrary code execution as the server's user. Leave it off.

In [ ]:
!chmod +x demo/build.sh Test/*.sh
!./demo/build.sh 2>&1 | tail -n 12

## 5 · Start the server (background)

The server blocks and listens forever, so it can't own a notebook cell — we launch it as a
background process and tail its log. It prints what it loaded, then listens on **2160**
(binary) and **2190** (HTTP).

In [ ]:
import time

# stop a previous instance if this cell is re-run
try:
    srv.terminate(); srv.wait(timeout=5)
except Exception:
    pass

srv = subprocess.Popen(['./home/bin/ziguratip'],
                       stdout=open('server.log', 'w'),
                       stderr=subprocess.STDOUT,
                       env=os.environ)
time.sleep(2.5)
print(open('server.log').read())
assert srv.poll() is None, 'server exited — see the log above'

## 6 · Reach the HTTP server from your browser

Colab does not expose raw ports to the internet, but it has a built-in port proxy. The cell
below returns a clickable HTTPS URL that front-ends port **2190**.

The binary protocol on 2160 is *not* browser-reachable — you would only use it from a
Connector client running inside this same VM.

Demo pages to try (append to the proxy URL): `/setup.zt` creates rows, `/catalog.zt` browses
them, `/lookup.zt`, `/bulk.zt`, `/report.zt`.

In [ ]:
import urllib.request, time

# Prove the server answers HERE before handing out a proxy URL.
# proxyPort() returns a URL whether or not anything is listening, so a dead
# server and a healthy one look identical from the browser: both render a
# blank page. Check locally first and say so plainly if it is not up.
code = None
for attempt in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:2190/', timeout=5) as r:
            code, body = r.status, r.read()
        break
    except Exception as e:
        err = e; time.sleep(1)

if code is None:
    print('SERVER IS NOT ANSWERING on 127.0.0.1:2190 --', err)
    print('--- server.log ---'); print(open('server.log').read()[-2000:])
    raise SystemExit('not starting the proxy: there is nothing behind it')

print(f'local check: HTTP {code}, {len(body)} bytes')
assert code == 200 and len(body) > 0, 'server answered but served nothing'

# Render it INSIDE the notebook first. This needs no proxy at all, so it
# works even when the proxy URL does not, and it is the fastest way to see
# that the server is really serving.
from IPython.display import HTML, display
display(HTML(body.decode('utf-8', 'replace')))

# Open the URL below in a browser TAB. That is the one that works properly:
# its host maps entirely to this server, so the relative links and images in
# the page -- href="setup.zt", src="zeytun.png" -- resolve against Zeytun.
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(2190)')
print('OPEN THIS IN A BROWSER TAB:')
print(' ', url)

# The iframe below is a liveness check, not a way to use the site. It is
# served through Colab's kernel proxy, whose base URL is localhost:8080, and
# relative links resolve against THAT -- so zeytun.png is fetched from
# Colab's own server instead of this one and 404s, and every link on the page
# points at 127.0.0.1:8080. The page itself renders; nothing under it will.
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(2190, height=420)

print('Demo:', url.rstrip('/') + '/setup.zt', '(run once, then /catalog.zt)')

## 6b · See the page with its images, without any proxy

The cell above depends on Colab routing your browser to this port, and that is where
things have gone wrong repeatedly: an iframe resolves the page's relative links
(`src="zeytun.png"`, `href="setup.zt"`) against Colab's own base URL rather than
Zeytun's, so the banner 404s and every link reads `127.0.0.1:8080`.

This cell sidesteps all of it. It fetches the page **and each asset it references**
over loopback, embeds them as `data:` URIs, and renders the result inline — so what
you see is what Zeytun actually served, images included, with no proxy in the path.

It is a viewer, not a way to browse: the `.zt` links stay relative and are meant to
be followed in a real tab.

In [ ]:
import urllib.request, base64, re, mimetypes
from IPython.display import HTML, display

BASE = 'http://127.0.0.1:2190'

def fetch(path):
    with urllib.request.urlopen(BASE + '/' + path.lstrip('/'), timeout=15) as r:
        return r.status, r.read(), r.headers.get('Content-Type', '')

def visit(path='/', quiet=False):
    """Fetch a page, embed everything it references, and show it.

    Nothing here goes through a proxy, so relative references resolve
    against this server rather than against Colab's own host -- which is
    what makes the banner load and the links read correctly.
    """
    status, body, ctype = fetch(path)
    html = body.decode('utf-8', 'replace')
    inlined = 0
    refs = sorted(set(re.findall(
        r'(?:src|href)="(?!https?:|//|#|mailto:|data:)([^"]+)"', html)))
    for ref in refs:
        if ref.endswith('.zt') or ref == '/':
            continue                     # a page: visit() it rather than embed it
        try:
            s, data, ct = fetch(ref)
            if s != 200: continue
            ct = ct or mimetypes.guess_type(ref)[0] or 'application/octet-stream'
            html = html.replace('"%s"' % ref,
                                '"data:%s;base64,%s"' % (ct, base64.b64encode(data).decode()))
            inlined += 1
        except Exception:
            pass
    if not quiet:
        print(f'{path}: HTTP {status}, {len(body)} bytes, {inlined} asset(s) embedded')
    display(HTML(html))
    return status

visit('/')

### Walking the demo

`visit()` takes any path, so the demo can be followed here without a browser:

```python
visit('/setup.zt')     # seed the catalogue -- run once
visit('/catalog.zt')   # the books and authors it created
visit('/lookup.zt')    # queries served from single-column indexes
visit('/report.zt')    # a two-column index
```

Running `setup.zt` a second time reports `unique key 'IDX_DEMO_AUTHORS_NAME'` —
that is the index refusing a duplicate author, which is the point of it.

In [ ]:
visit('/catalog.zt')

## 6d · Does the browser's request even reach the server?

Four rounds of "blank page" have all turned on the hop between a browser and
this port, never on the server, and guessing which hop has been expensive. This
settles it in one run.

Run the cell, open the URL it prints in a browser tab, wait for it to fail, then
run the **next** cell. It reports whether anything new appeared in `server.log`.

- **New lines** → the request arrived. Whatever Zeytun answered is ours to fix,
  and refusals now carry a body, so view-source will say what happened.
- **Nothing** → the request never arrived. The 404 is Colab's own edge — its
  `404 page not found` is the stock body Go's net/http writes, and nothing this
  server does can change it. Use `visit()` above instead.

In [ ]:
import os, urllib.request

LOG = '/content/ZiguratIP/server.log'

status, body, _ = fetch('/')
print(f'loopback check: HTTP {status}, {len(body)} bytes  <- the server itself is fine')

before = os.path.getsize(LOG) if os.path.exists(LOG) else 0
globals()['_log_mark'] = before
print(f'server.log is {before} bytes right now')

from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(2190)')
print()
print('NOW open this in a browser tab, let it finish (or fail), then run the next cell:')
print(' ', url)

In [ ]:
mark = globals().get('_log_mark', 0)
size = os.path.getsize(LOG) if os.path.exists(LOG) else 0

if size > mark:
    with open(LOG) as f:
        f.seek(mark)
        new = f.read()
    print('THE REQUEST REACHED ZEYTUN. It logged:')
    print(new)
else:
    print('NOTHING NEW IN THE LOG -- the request never reached this server.')
    print()
    print("Colab's edge answered before the request got here, so no change to")
    print('ZiguratIP can affect it. Use visit() above to work with the demo.')

## 6e · (Optional) a real URL, through a Cloudflare tunnel

Colab's proxy never delivers the request — the diagnostic above shows nothing arriving.
A tunnel goes the other way: `cloudflared` opens an outbound connection to Cloudflare and
they hand you a public hostname that forwards back down it. Nothing has to reach *in*,
so nothing about Colab's routing matters, and because the whole host maps to Zeytun the
page's relative links and images resolve properly.

> 🔓 **This puts the server on the public internet.** Anyone with the URL can reach it,
> and it is the server described at the top of this notebook: cross-site scripting by
> construction, no write-ahead log, zlib 1.2.11. There is no authentication in front of
> it. Use it to look at the demo, keep it short, and stop it when you are done — the URL
> is unguessable but it is not a secret, and a quick tunnel has no uptime guarantee and
> no access control whatsoever.

Set `TUNNEL = True` to opt in. It stays off by default deliberately.

In [ ]:
import os, re, subprocess, time

TUNNEL = False   # set True to expose this server publicly, having read the warning

CF  = '/content/cloudflared'
LOG = '/content/cloudflared.log'

if not TUNNEL:
    print('TUNNEL is off. Nothing is exposed.')
else:
    if not os.path.exists(CF):
        print('fetching cloudflared ...')
        subprocess.run(['curl','-sSL','-o',CF,
                        'https://github.com/cloudflare/cloudflared/releases/latest/download/'
                        'cloudflared-linux-amd64'], check=True)
        os.chmod(CF, 0o755)

    # A quick tunnel: no account, no signup, and no access control either.
    with open(LOG, 'w') as log:
        tunnel = subprocess.Popen(
            [CF, 'tunnel', '--url', 'http://localhost:2190', '--no-autoupdate'],
            stdout=log, stderr=subprocess.STDOUT)
    globals()['_tunnel'] = tunnel

    # cloudflared prints the hostname in a banner once Cloudflare has assigned it.
    url = None
    for _ in range(45):
        if tunnel.poll() is not None:
            break
        try:
            found = re.findall(r'https://[a-z0-9][a-z0-9-]*\.trycloudflare\.com',
                               open(LOG).read())
            if found:
                url = found[0]; break
        except FileNotFoundError:
            pass
        time.sleep(2)

    if url:
        print('Open this in a browser:')
        print(' ', url)
        print()
        print('  demo:', url + '/setup.zt', '(once), then', url + '/catalog.zt')
        print()
        print('Stop it with:  _tunnel.terminate()')
    else:
        # Say why rather than leaving a dead cell -- the usual causes are an
        # egress policy blocking api.trycloudflare.com, or the server not
        # listening on 2190.
        print('NO TUNNEL URL. cloudflared said:')
        try:
            for line in open(LOG).read().splitlines()[-12:]:
                print('  ', line[:160])
        except FileNotFoundError:
            print('   (no log written at all)')

## 7 · (Optional) snapshot data back to Drive

Only meaningful when `PERSIST = True` **and** after a clean stop (next cell). ZiguratIP has no
write-ahead log, so a snapshot taken while the server is mid-write can be inconsistent — stop
the server first, then run this.

In [ ]:
if PERSIST:
    !mkdir -p "$(dirname "$DRIVE_BACKUP")"
    !rm -rf "$DRIVE_BACKUP"
    !cp -a $ZIGURATIP_HOME/data "$DRIVE_BACKUP"
    print('snapshot written to', DRIVE_BACKUP)
else:
    print('PERSIST is off — nothing to save.')

## 8 · Stop the server

In [ ]:
try:
    srv.terminate(); srv.wait(timeout=5)
    print('server stopped.')
except Exception as e:
    print('nothing running:', e)

## Notes & limits

- **Durability:** the storage engine has no WAL/fsync, so a hard runtime kill mid-commit can
  corrupt the store. Snapshot to Drive only after a clean stop.
- **Concurrency:** thread-per-connection on a pool of 64, backlog 64. It was 5, which one
  visitor could saturate on their own — six parallel connections is what a browser opens for
  a single page, and the seventh waited on the first six.
- **TLS:** the server speaks interoperable TLS now — ECDHE and AEAD only, static RSA refused
  outright, TLS 1.3 negotiated, and a browser needs no certificate of its own if the port is
  set `TLS_CLIENT_AUTH: NONE`. Leave `HTTP/TLS_MODE: FALSE` **in Colab** anyway: the proxy
  speaks plain HTTP to your port and terminates TLS itself, so turning it on here just means
  two layers talking past each other. On a real host, turn it on.
- **Reaching it from a browser in Colab:** you cannot, and it is not this server's doing. The
  diagnostic above confirmed the request never arrives — Colab's edge answers `404 page not
  found`, which is the stock body Go's net/http writes, before anything gets here. `visit()`
  is the way to use the demo from a Colab session.
- **Drive:** never set `HOME_PATH`/data onto the `/content/drive` mount — FUSE breaks the
  pager's random in-place writes and `dlopen`. Local disk for running, Drive for snapshots.
- **Security:** see `doc/outstanding.md`. The short version: XSS by construction, no
  write-ahead log, and vendored zlib 1.2.11. Treat every run as disposable.